# Greenwashing Detector - Text Cleaning and Analysis

This notebook demonstrates how to use the Greenwashing Detector to clean and analyze text for potential greenwashing claims.

## What is Greenwashing?

Greenwashing is when companies make misleading claims about the environmental benefits of their products or services. Common tactics include:
- Using vague terms like "eco-friendly" or "natural" without substantiation
- Making claims that sound environmental but are meaningless
- Highlighting minor green attributes while ignoring major environmental impacts

## Setup

First, let's import the necessary libraries and modules.

In [ ]:
import sys
import os

# Add the parent directory to the path to import our modules
sys.path.insert(0, os.path.abspath('..'))

from src.text_cleaner import clean_text, tokenize_text, extract_keywords
from src.scoring import calculate_greenwashing_score, analyze_text, get_keyword_list

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set up plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All modules imported successfully!")

## 1. Text Cleaning

Let's start by exploring the text cleaning functionality.

In [ ]:
# Example text with potential greenwashing
sample_text = """
Our new eco-friendly product is made with 100% natural ingredients! 
It's chemical-free, biodegradable, and good for the environment. 
Join us in making the planet better with our sustainable, green solution.
Visit www.example.com for more info!
"""

print("Original text:")
print(sample_text)
print("\n" + "="*50 + "\n")

# Clean the text
cleaned = clean_text(sample_text)
print("Cleaned text:")
print(cleaned)

In [ ]:
# Clean text with stopwords removed
cleaned_no_stopwords = clean_text(sample_text, remove_stopwords=True)
print("Cleaned text (stopwords removed):")
print(cleaned_no_stopwords)

In [ ]:
# Extract keywords
keywords = extract_keywords(sample_text)
print(f"Extracted keywords ({len(keywords)}):")
print(keywords)

## 2. Greenwashing Detection

Now let's analyze text for potential greenwashing.

In [ ]:
# Analyze the sample text
print(analyze_text(sample_text))

In [ ]:
# Get detailed scoring information
result = calculate_greenwashing_score(sample_text)
print(f"Score: {result['score']}/100")
print(f"Risk Level: {result['risk_level']}")
print(f"\nMatched Keywords: {len(result['matched_keywords'])}")
for match in result['matched_keywords']:
    print(f"  - {match['keyword']}: {match['count']}x (weight: {match['weight']})")

## 3. Analyzing Multiple Texts

Let's analyze several different product claims.

In [ ]:
# Create a dataset of sample product claims
claims = [
    {
        'id': 1,
        'product': 'Product A',
        'claim': 'Our eco-friendly, all natural cleaning solution is safe for the planet!'
    },
    {
        'id': 2,
        'product': 'Product B',
        'claim': 'Made with recycled materials. Certified by environmental standards.'
    },
    {
        'id': 3,
        'product': 'Product C',
        'claim': 'Chemical-free, 100% green, sustainable, organic, earth-friendly packaging.'
    },
    {
        'id': 4,
        'product': 'Product D',
        'claim': 'Manufactured with energy-efficient processes. Reduced carbon footprint.'
    },
    {
        'id': 5,
        'product': 'Product E',
        'claim': 'High-quality materials ensure durability and long product life.'
    }
]

# Analyze each claim
results = []
for claim in claims:
    score_result = calculate_greenwashing_score(claim['claim'])
    results.append({
        'Product': claim['product'],
        'Claim': claim['claim'][:50] + '...' if len(claim['claim']) > 50 else claim['claim'],
        'Score': score_result['score'],
        'Risk Level': score_result['risk_level'],
        'Keywords Found': len(score_result['matched_keywords'])
    })

# Create a DataFrame
df_results = pd.DataFrame(results)
df_results

## 4. Visualizing Results

Let's create some visualizations to better understand the results.

In [ ]:
# Bar plot of greenwashing scores
plt.figure(figsize=(10, 6))
colors = ['green' if score < 5 else 'orange' if score < 15 else 'red' 
          for score in df_results['Score']]
plt.bar(df_results['Product'], df_results['Score'], color=colors, alpha=0.7)
plt.axhline(y=5, color='orange', linestyle='--', label='Medium Risk Threshold')
plt.axhline(y=15, color='red', linestyle='--', label='High Risk Threshold')
plt.xlabel('Product')
plt.ylabel('Greenwashing Score')
plt.title('Greenwashing Scores by Product')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of risk levels
risk_counts = df_results['Risk Level'].value_counts()
plt.figure(figsize=(8, 6))
colors_pie = {'Low': 'green', 'Medium': 'orange', 'High': 'red'}
plt.pie(risk_counts.values, labels=risk_counts.index, autopct='%1.1f%%',
        colors=[colors_pie.get(level, 'gray') for level in risk_counts.index])
plt.title('Distribution of Risk Levels')
plt.axis('equal')
plt.show()

## 5. View Detection Keywords

Let's see what keywords the detector is looking for.

In [ ]:
# Get the keyword lists
keywords_dict = get_keyword_list()

print("Greenwashing Keywords:")
print("="*50)
for keyword, weight in sorted(keywords_dict['greenwashing'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {keyword}: weight={weight}")

print("\nVague/Misleading Terms:")
print("="*50)
for term, weight in sorted(keywords_dict['vague'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {term}: weight={weight}")

## 6. Try Your Own Text

Now it's your turn! Add your own text to analyze below.

In [ ]:
# Enter your own text here
your_text = """
Enter your product claim or marketing text here to analyze it for greenwashing.
"""

print(analyze_text(your_text))

## Next Steps

This is a basic keyword-based greenwashing detector. To improve it, you could:

1. **Add more keywords**: Expand the keyword lists with more greenwashing terms
2. **Context analysis**: Check if environmental claims are backed by specific data or certifications
3. **Load real data**: Import CSV files with product claims from the `/data` folder
4. **Improve scoring**: Adjust weights based on your domain knowledge
5. **Export results**: Save analysis results to CSV files for further review

Remember: This is a simple starter tool. Real greenwashing detection would require:
- Verification of claims against actual data
- Understanding of industry-specific environmental standards
- Context about the company and product
- Expert review of environmental claims